# Expected (Noisy Runs) vs Observed (Baseline Real-vs-Noisy TVD)

This notebook generates `slot/4.png` using the latest report JSON under `artifacts/*/*.json`:
- **Expected**: repeated local noisy runs via `QuantumExecutor` (`noisy_simulator` vs `ideal_simulator`) on a reconstructed circuit from the report
- **Observed**: baseline loss from the same report (`ddmin_log` action = `baseline`)

It also saves trial-level data to `slot/4_noisy_trials.csv`.

In [ ]:
from __future__ import annotations

import csv
import json
import re
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qiskit import QuantumCircuit
from quantum.executor import QuantumExecutor
from quantum.metrics import calculate_tvd

ARTIFACTS_ROOT = REPO_ROOT / 'artifacts'
REPORT_FILES = sorted(ARTIFACTS_ROOT.glob('*/*.json'), key=lambda p: p.stat().st_mtime)
if not REPORT_FILES:
    raise FileNotFoundError(f'No report JSON found under: {ARTIFACTS_ROOT}')

REPORT_FILE = REPORT_FILES[-1]
ARTIFACT_DIR = REPORT_FILE.parent
OUT_PNG = REPO_ROOT / 'slot' / '4.png'
OUT_CSV = REPO_ROOT / 'slot' / '4_noisy_trials.csv'
CONFIG_FILE = REPO_ROOT / 'quantum_config.json'

SHOTS = 1024
TRIALS = 1

print('report file:', REPORT_FILE)
print('artifact dir:', ARTIFACT_DIR)
print('output image:', OUT_PNG)
print('output csv:', OUT_CSV)
print('config:', CONFIG_FILE)

In [ ]:
def load_report(report_path: Path) -> dict:
    with report_path.open('r', encoding='utf-8') as f:
        return json.load(f)


def extract_baseline_loss(report: dict) -> float:
    for e in report.get('ddmin_log', []):
        if e.get('action') == 'baseline' and e.get('loss') is not None:
            return float(e['loss'])
    raise RuntimeError('Cannot find baseline loss in ddmin_log')


def _ordered_instructions(report: dict) -> list[dict]:
    rows = []
    for seg in report.get('segments_info', []):
        for inst in seg.get('instructions', []):
            if not isinstance(inst, dict):
                continue
            if not inst.get('operation'):
                continue
            idx = inst.get('index')
            order_key = idx if isinstance(idx, int) else 10**9
            rows.append((order_key, inst))
    rows.sort(key=lambda x: x[0])
    return [inst for _, inst in rows]


def build_circuit_from_report(report: dict):
    ordered = _ordered_instructions(report)
    if not ordered:
        raise RuntimeError('No instructions found in report segments_info')

    used_qubits = sorted({int(q) for inst in ordered for q in (inst.get('qubits') or [])})
    if not used_qubits:
        raise RuntimeError('No qubits found in report instructions')

    qmap = {q: i for i, q in enumerate(used_qubits)}
    qc = QuantumCircuit(len(used_qubits), len(used_qubits))
    unsupported = set()

    for inst in ordered:
        name = str(inst.get('operation') or '').lower()
        qargs = [qmap[int(q)] for q in (inst.get('qubits') or [])]
        params = inst.get('params') or []

        if name == 'rz' and len(qargs) == 1:
            qc.rz(float(params[0]) if params else 0.0, qargs[0])
        elif name == 'sx' and len(qargs) == 1:
            qc.sx(qargs[0])
        elif name == 'x' and len(qargs) == 1:
            qc.x(qargs[0])
        elif name == 'cz' and len(qargs) == 2:
            qc.cz(qargs[0], qargs[1])
        elif name == 'cx' and len(qargs) == 2:
            qc.cx(qargs[0], qargs[1])
        elif name == 'ecr' and len(qargs) == 2:
            qc.ecr(qargs[0], qargs[1])
        elif name in {'barrier', 'measure'}:
            continue
        else:
            unsupported.add(name or 'UNKNOWN')

    qc.measure(range(qc.num_qubits), range(qc.num_clbits))
    return qc, used_qubits, len(ordered), sorted(unsupported)


def run_expected_from_noisy_runs(qe: QuantumExecutor, qc: QuantumCircuit, shots: int, trials: int):
    trial_rows = []
    vals = []
    for i in range(1, trials + 1):
        ideal = qe.run_circuit(qc, execution_type='ideal_simulator', shots=shots)
        noisy = qe.run_circuit(qc, execution_type='noisy_simulator', shots=shots)
        tvd, _ = calculate_tvd(noisy.get('counts', {}), ideal.get('counts', {}))
        vals.append(float(tvd))
        trial_rows.append({
            'trial': i,
            'shots': shots,
            'expected_noisy_tvd': float(tvd),
            'ideal_counts': json.dumps(ideal.get('counts', {}), ensure_ascii=False),
            'noisy_counts': json.dumps(noisy.get('counts', {}), ensure_ascii=False),
        })
    mean_v = statistics.fmean(vals)
    return mean_v, trial_rows


def save_trials_csv(rows, out_csv: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    fields = ['trial', 'shots', 'expected_noisy_tvd', 'ideal_counts', 'noisy_counts']
    with out_csv.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow(r)

In [ ]:
report = load_report(REPORT_FILE)
observed = extract_baseline_loss(report)
qc, qubits, n_ops, unsupported_ops = build_circuit_from_report(report)

print('selected report:', REPORT_FILE.name)
print('observed baseline (real-vs-noisy TVD):', observed)
print('reconstructed operations:', n_ops)
print('used physical qubits in report:', qubits)
if unsupported_ops:
    print('unsupported ops skipped:', unsupported_ops)

qe = QuantumExecutor(config_file=str(CONFIG_FILE))
expected_mean, trial_rows = run_expected_from_noisy_runs(qe, qc, shots=SHOTS, trials=TRIALS)
save_trials_csv(trial_rows, OUT_CSV)

deviation = observed / expected_mean if expected_mean > 0 else float('inf')

m = re.search(r'(\d{8})_(\d{6})', REPORT_FILE.name)
date_str = f"{m.group(1)[0:4]}-{m.group(1)[4:6]}-{m.group(1)[6:8]}" if m else 'N/A'
name_parts = REPORT_FILE.stem.split('_')
backend_name = '_'.join(name_parts[:2]) if len(name_parts) >= 2 else 'N/A'

fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.10, 0.23, 0.82, 0.65])
vals_pct = [expected_mean * 100, observed * 100]
bars = ax.bar(
    ['Expected\n(noisy simulator runs)', 'Observed\n(real backend baseline)'],
    vals_pct,
    color=['steelblue', 'tomato'],
)
ax.set_ylabel('Error Rate (%)')
ax.set_title('Expected vs Observed Error Rate')
ax.grid(axis='y', linestyle='--', alpha=0.3)
for b, v in zip(bars, vals_pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{v:.2f}%', ha='center', va='bottom', fontsize=10)

fig.suptitle('Validation of High-Loss Circuit Behavior', fontsize=14, fontweight='bold')
caption = f'Date: {date_str} | Backend: {backend_name} | Report: {REPORT_FILE.name}'
fig.text(0.5, 0.08, caption, ha='center', va='center', fontsize=9)

OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=180)
plt.show()

print('saved image:', OUT_PNG)
print('saved trials:', OUT_CSV)
print('expected mean:', expected_mean)
print('observed     :', observed)
print('deviation x  :', deviation)